<a href="https://colab.research.google.com/github/soleildayana/Apophis-Asteroid-Project/blob/main/analisis/nb7_perturbaciones_gauss.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Notebook 7 — Ecuaciones de Gauss: Variación de Elementos Orbitales
## Perturbaciones gravitacionales sobre Apophis durante el encuentro 2029

**Autor:** Soleil Dayana Niño Murcia — 1033097666  
**Curso:** Mecánica Celeste  
**Fecha:** Mayo 2026

---

> **Objetivo:** Modelar cómo cambian los elementos orbitales de Apophis (especialmente
> la inclinación $i$ y el nodo ascendente $\Omega$) durante el encuentro cercano de 2029,
> usando las ecuaciones de Gauss-Lagrange (VOP), y comparar con los resultados de la
> integración N-cuerpos del modelo principal.

## 1. Teoría: Variación de Parámetros Orbitales (VOP)

### 1.1 Descomposición de la fuerza perturbadora

La fuerza perturbadora $\mathbf{F}_{\text{pert}}$ se proyecta en la terna local del objeto:

$$\mathbf{F}_{\text{pert}} = R\,\hat{r} + S\,\hat{s} + W\,\hat{w}$$

donde:
- $\hat{r}$: dirección radial (hacia afuera)
- $\hat{s} = \hat{w} \times \hat{r}$: dirección transversal (en el plano orbital, perpendicular a $\hat{r}$)
- $\hat{w} = \hat{h} = (\mathbf{r} \times \mathbf{v}) / |\mathbf{r} \times \mathbf{v}|$: normal al plano orbital

### 1.2 Ecuaciones de Gauss (forma VOP)

Las ecuaciones de variación de parámetros de Gauss para los elementos orbitales osciladores:

$$\frac{da}{dt} = \frac{2a^2}{h}\left[e\sin f \cdot R + \frac{p}{r}S\right]$$

$$\frac{de}{dt} = \frac{1}{h}\left[p\sin f \cdot R + ((p+r)\cos f + re)S\right]$$

$$\frac{di}{dt} = \frac{r\cos(\omega + f)}{h}\,W$$

$$\frac{d\Omega}{dt} = \frac{r\sin(\omega + f)}{h\sin i}\,W$$

$$\frac{d\omega}{dt} = \frac{1}{he}\left[-p\cos f \cdot R + (p+r)\sin f \cdot S\right] - \cos i \frac{d\Omega}{dt}$$

donde $r = p/(1 + e\cos f)$, $h = \sqrt{\mu p}$.

### 1.3 Variables de Delaunay (formalismo canónico)

Las ecuaciones de Gauss tienen un equivalente canónico en las **variables de Delaunay**:

$$L = \sqrt{\mu a}, \quad G = L\sqrt{1-e^2}, \quad H = G\cos i$$

con conjugadas $l = M$ (anomalía media), $g = \omega$, $h = \Omega$.

En estas variables, el Hamiltoniano toma la forma $\mathcal{H} = \mathcal{H}_0(L) + \mathcal{H}_1(L,G,H,l,g,h)$,
donde $\mathcal{H}_1$ es la perturbación. Las ecuaciones de movimiento son las de Hamilton,
lo que facilita la identificación de simetrías e invariantes adiabáticos.

In [ ]:
%pip install -Uq pymcel

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from scipy.integrate import solve_ivp

import pymcel as pc
from pymcel import constantes as const

print('Librerías cargadas correctamente.')

## 2. Configuración: unidades y datos del encuentro

In [ ]:
# ── Constantes ─────────────────────────────────────────────────────────────
AU_km    = 149_597_870.7
AU_m     = AU_km * 1e3
M_sun_kg = 1.989e30
G_SI     = 6.674e-11
UT_s     = np.sqrt(AU_m**3 / (G_SI * M_sun_kg))
UT_days  = UT_s / 86_400.0
vel_unit = AU_km / UT_s

GM_sun_km3s2   = 1.32712440018e11
GM_earth_km3s2 = 3.986004418e5
GM_moon_km3s2  = 4.9048695e3

# Masas canónicas
mu_sol    = 1.0
m_tierra  = GM_earth_km3s2 / GM_sun_km3s2
m_luna    = GM_moon_km3s2  / GM_sun_km3s2
m_apophis = 1e-20

print(f'UT = {UT_days:.4f} días  |  1 AU/UT = {vel_unit:.4f} km/s')
print(f'm_Tierra  = {m_tierra:.4e}')
print(f'm_Luna    = {m_luna:.4e}')

# ── Ventana del encuentro ──────────────────────────────────────────────────
# ±60 días alrededor del 2029-04-13
INICIO_STR = '2029-02-12'
FIN_STR    = '2029-06-12'
ENCUENTRO_STR = '2029-04-13'

## 3. Descarga de condiciones iniciales y configuración del sistema N-cuerpos

In [ ]:
cuerpos_hz = [
    ('Sol',     '10'),
    ('Tierra',  '399'),
    ('Luna',    '301'),
    ('Apophis', '99942'),
]

estados_raw = {}
for nombre, id_hz in cuerpos_hz:
    print(f'Consultando {nombre:8s}...', end=' ')
    tabla, jd, estado = pc.consulta_horizons(
        id       = id_hz,
        location = '@0',
        epochs   = INICIO_STR,
    )
    r_vec = estado[:3]
    v_vec = estado[3:] * UT_days  # AU/día → AU/UT
    estados_raw[nombre] = {'r': r_vec, 'v': v_vec}
    print(f'OK | r = {np.linalg.norm(r_vec):.3f} AU')

masas = {
    'Sol': mu_sol, 'Tierra': m_tierra,
    'Luna': m_luna, 'Apophis': m_apophis,
}

sistema = [
    dict(m=masas[n], r=list(estados_raw[n]['r']), v=list(estados_raw[n]['v']))
    for n in ['Sol', 'Tierra', 'Luna', 'Apophis']
]

print(f'\nSistema con {len(sistema)} cuerpos configurado.')

## 4. Integración N-cuerpos en la ventana del encuentro (±60 días)

In [ ]:
# Ventana: 120 días en UT
DURACION_DIAS = 120
N_PASOS_NC    = 2000

t_fin_UT  = DURACION_DIAS / UT_days
ts_nc     = np.linspace(0.0, t_fin_UT, N_PASOS_NC)
dt_nc_h   = (ts_nc[1] - ts_nc[0]) * UT_days * 24

print(f'Integrando {DURACION_DIAS} días ({t_fin_UT:.4f} UT) con {N_PASOS_NC} pasos (Δt ≈ {dt_nc_h:.1f} h)...')

rs_nc, vs_nc, _, _, constantes_nc = pc.ncuerpos_solucion(sistema, ts_nc)

print(f'✓ Integración completa.')
print(f'  Dimensiones: rs={rs_nc.shape}, vs={vs_nc.shape}')

# Índices de los cuerpos en el sistema
IDX = {'Sol': 0, 'Tierra': 1, 'Luna': 2, 'Apophis': 3}

# Trayectorias de Apophis y Tierra
r_apophis_nc = rs_nc[IDX['Apophis']]  # shape (N_PASOS_NC, 3)
v_apophis_nc = vs_nc[IDX['Apophis']]
r_tierra_nc  = rs_nc[IDX['Tierra']]
r_sol_nc     = rs_nc[IDX['Sol']]
r_luna_nc    = rs_nc[IDX['Luna']]

# Tiempo en días desde el inicio
t_dias_nc = ts_nc * UT_days

## 5. Extracción de la fuerza perturbadora terrestre sobre Apophis

La fuerza perturbadora es la aceleración gravitacional de la Tierra (y Luna) sobre Apophis,
excluyendo la fuerza dominante del Sol (que genera la órbita kepleriana de referencia).

In [ ]:
# Calcular la fuerza perturbadora en cada paso temporal
# F_pert = a_Tierra_sobre_Apophis + a_Luna_sobre_Apophis

def aceleracion_grav(r_objeto, r_fuente, masa_fuente):
    """Aceleración gravitacional de r_fuente sobre r_objeto (G=1)."""
    delta_r = r_fuente - r_objeto
    dist    = np.linalg.norm(delta_r)
    return masa_fuente * delta_r / dist**3

F_pert = np.zeros_like(r_apophis_nc)  # shape (N_PASOS_NC, 3)

for k in range(N_PASOS_NC):
    acc_tierra = aceleracion_grav(r_apophis_nc[k], r_tierra_nc[k], m_tierra)
    acc_luna   = aceleracion_grav(r_apophis_nc[k], r_luna_nc[k],   m_luna)
    F_pert[k]  = acc_tierra + acc_luna

# Magnitud de la perturbación
F_pert_mag = np.linalg.norm(F_pert, axis=1)
print(f'Fuerza perturbadora calculada.')
print(f'  Máximo: {F_pert_mag.max():.4e} AU/UT²  en el paso {F_pert_mag.argmax()}')
print(f'  Mínimo: {F_pert_mag.min():.4e} AU/UT²')

## 6. Proyección en la terna {r̂, ŝ, ŵ}

In [ ]:
# Vectores de la terna local de Apophis relativa al Sol
R_comp = np.zeros(N_PASOS_NC)
S_comp = np.zeros(N_PASOS_NC)
W_comp = np.zeros(N_PASOS_NC)

# Posición y velocidad de Apophis relativa al Sol (baricéntrico ≈ Sol para simplificar)
r_apo_rel_sol = r_apophis_nc - r_sol_nc
v_apo_rel_sol = v_apophis_nc - vs_nc[IDX['Sol']]

for k in range(N_PASOS_NC):
    r_k = r_apo_rel_sol[k]
    v_k = v_apo_rel_sol[k]

    # Terna ortogonal
    r_mag = np.linalg.norm(r_k)
    r_hat = r_k / r_mag                          # radial

    h_k   = np.cross(r_k, v_k)                   # momento angular
    h_mag = np.linalg.norm(h_k)
    w_hat = h_k / h_mag                          # normal al plano

    s_hat = np.cross(w_hat, r_hat)               # transversal
    s_hat /= np.linalg.norm(s_hat)

    # Proyecciones de la perturbación
    R_comp[k] = np.dot(F_pert[k], r_hat)
    S_comp[k] = np.dot(F_pert[k], s_hat)
    W_comp[k] = np.dot(F_pert[k], w_hat)

print('Proyecciones RSW calculadas.')
print(f'  |R| máximo: {np.abs(R_comp).max():.4e}')
print(f'  |S| máximo: {np.abs(S_comp).max():.4e}')
print(f'  |W| máximo: {np.abs(W_comp).max():.4e}')

## 7. Integración de las ecuaciones de Gauss

Integramos las ecuaciones de Gauss para $di/dt$ y $d\Omega/dt$, alimentadas con
las componentes $W(t)$ interpoladas de la integración N-cuerpos.

In [ ]:
from scipy.interpolate import interp1d

# Elementos orbitales osciladores de Apophis en cada paso
a_nc  = np.zeros(N_PASOS_NC)
e_nc  = np.zeros(N_PASOS_NC)
i_nc  = np.zeros(N_PASOS_NC)
Omega_nc = np.zeros(N_PASOS_NC)
omega_nc = np.zeros(N_PASOS_NC)
f_nc     = np.zeros(N_PASOS_NC)

mu_sol_val = 1.0
for k in range(N_PASOS_NC):
    r_k = r_apo_rel_sol[k]
    v_k = v_apo_rel_sol[k]
    try:
        estado_k = np.concatenate([r_k, v_k])
        elems_k  = pc.estado_a_elementos(mu_sol_val, estado_k)
        p_k, e_k, i_k, O_k, o_k, f_k = elems_k
        a_nc[k]     = p_k / (1 - e_k**2)
        e_nc[k]     = e_k
        i_nc[k]     = i_k
        Omega_nc[k] = O_k
        omega_nc[k] = o_k
        f_nc[k]     = f_k
    except Exception:
        a_nc[k] = a_nc[k-1] if k > 0 else 0.92
        e_nc[k] = e_nc[k-1] if k > 0 else 0.191
        i_nc[k] = i_nc[k-1] if k > 0 else np.radians(3.34)
        Omega_nc[k] = Omega_nc[k-1] if k > 0 else 0

print('Elementos orbitales osciladores calculados.')

# Desenvolver ángulos antes de interpolar para evitar saltos espurios en 0↔2π
omega_nc_unwrapped = np.unwrap(omega_nc)
f_nc_unwrapped     = np.unwrap(f_nc)

# Interpoladores para usar en la integración de Gauss
W_interp     = interp1d(ts_nc, W_comp,             kind='linear', fill_value='extrapolate')
R_interp     = interp1d(ts_nc, R_comp,             kind='linear', fill_value='extrapolate')
S_interp     = interp1d(ts_nc, S_comp,             kind='linear', fill_value='extrapolate')
a_interp     = interp1d(ts_nc, a_nc,               kind='linear', fill_value='extrapolate')
e_interp     = interp1d(ts_nc, e_nc,               kind='linear', fill_value='extrapolate')
omega_interp = interp1d(ts_nc, omega_nc_unwrapped, kind='linear', fill_value='extrapolate')
f_interp     = interp1d(ts_nc, f_nc_unwrapped,     kind='linear', fill_value='extrapolate')

In [ ]:
def gauss_equations(t, Y):
    """
    Ecuaciones de Gauss para i(t) y Ω(t).
    Y = [i, Omega]
    """
    i_val = Y[0]
    sin_i = np.sin(i_val)
    if abs(sin_i) < 1e-12:
        sin_i = 1e-12

    W_val   = W_interp(t)
    a_val   = a_interp(t)
    e_val   = e_interp(t)
    f_val   = f_interp(t)
    omega_v = omega_interp(t)

    p_val   = a_val * (1 - e_val**2)
    r_val   = p_val / (1 + e_val * np.cos(f_val))
    h_val   = np.sqrt(mu_sol_val * p_val)

    arg_lat = omega_v + f_val  # argumento de latitud

    # di/dt = (r * cos(omega+f) / h) * W
    di_dt = (r_val * np.cos(arg_lat) / h_val) * W_val

    # dΩ/dt = (r * sin(omega+f) / (h * sin(i))) * W
    dOmega_dt = (r_val * np.sin(arg_lat) / (h_val * sin_i)) * W_val

    return [di_dt, dOmega_dt]


# Condiciones iniciales: elementos del primer paso N-cuerpos
Y0_gauss = [i_nc[0], Omega_nc[0]]

print(f'Condiciones iniciales Gauss:')
print(f'  i(0)     = {np.degrees(Y0_gauss[0]):.4f}°')
print(f'  Ω(0)     = {np.degrees(Y0_gauss[1]):.4f}°')

print(f'\nIntegrando ecuaciones de Gauss...')
sol_gauss = solve_ivp(
    gauss_equations,
    t_span  = (ts_nc[0], ts_nc[-1]),
    y0      = Y0_gauss,
    t_eval  = ts_nc,
    method  = 'RK45',
    rtol    = 1e-9,
    atol    = 1e-12,
)

print(f'  Estado: {sol_gauss.message}')
i_gauss     = sol_gauss.y[0]
Omega_gauss = sol_gauss.y[1]
print(f'  i_final  Gauss    = {np.degrees(i_gauss[-1]):.6f}°')
print(f'  i_final  N-cuerpos= {np.degrees(i_nc[-1]):.6f}°')
print(f'  Ω_final  Gauss    = {np.degrees(Omega_gauss[-1]):.6f}°')
print(f'  Ω_final  N-cuerpos= {np.degrees(Omega_nc[-1]):.6f}°')

## 8. Comparación: Gauss vs. N-cuerpos

In [ ]:
# Tiempo en días desde el inicio
t_dias_plot = ts_nc * UT_days

# Día del encuentro (offset desde INICIO = 2029-02-12)
from datetime import date
d_inicio    = date(2029, 2, 12)
d_encuentro = date(2029, 4, 13)
t_encuentro_dias = (d_encuentro - d_inicio).days

fig, axes = plt.subplots(2, 1, figsize=(12, 9), sharex=True)

# ─── Panel 1: inclinación ──────────────────────────────────────────────────
axes[0].plot(t_dias_plot, np.degrees(i_nc),
             'b-', linewidth=2, label='N-cuerpos (osciladores)')
axes[0].plot(t_dias_plot, np.degrees(i_gauss),
             'r--', linewidth=1.5, label='Ecuaciones de Gauss')
axes[0].axvline(t_encuentro_dias, color='green', linestyle='-.',
                linewidth=2, label=f'Encuentro (día {t_encuentro_dias})')
axes[0].set_ylabel('Inclinación $i$ (°)', fontsize=11)
axes[0].legend(fontsize=9)
axes[0].grid(alpha=0.3)
axes[0].set_title('Variación de $i$ y $\\Omega$ de Apophis durante el encuentro 2029\n'
                  'Ecuaciones de Gauss vs. Integración N-cuerpos', fontsize=11)

# ─── Panel 2: nodo ascendente ──────────────────────────────────────────────
axes[1].plot(t_dias_plot, np.degrees(Omega_nc),
             'b-', linewidth=2, label='N-cuerpos (osciladores)')
axes[1].plot(t_dias_plot, np.degrees(Omega_gauss),
             'r--', linewidth=1.5, label='Ecuaciones de Gauss')
axes[1].axvline(t_encuentro_dias, color='green', linestyle='-.',
                linewidth=2, label=f'Encuentro (día {t_encuentro_dias})')
axes[1].set_ylabel('Nodo ascendente $\\Omega$ (°)', fontsize=11)
axes[1].set_xlabel(f'Días desde {INICIO_STR}', fontsize=11)
axes[1].legend(fontsize=9)
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.savefig('gauss_vs_ncuerpos.png', dpi=120, bbox_inches='tight')
plt.show()
print('Figura guardada: gauss_vs_ncuerpos.png')

In [ ]:
# Gráfica de la fuerza perturbadora y distancia Tierra-Apophis
d_tierra_apophis = np.linalg.norm(r_apophis_nc - r_tierra_nc, axis=1)
R_Earth_AU = 6_371.0 / AU_km

fig, axes2 = plt.subplots(3, 1, figsize=(12, 10), sharex=True)

# Panel 1: distancia Tierra-Apophis
axes2[0].plot(t_dias_plot, d_tierra_apophis / R_Earth_AU,
              'k-', linewidth=1.5)
axes2[0].axvline(t_encuentro_dias, color='red', linestyle='--', linewidth=1.5)
axes2[0].set_ylabel('Distancia T-A ($R_\\oplus$)', fontsize=10)
axes2[0].set_yscale('log')
axes2[0].grid(alpha=0.3)
axes2[0].set_title('Fuerza perturbadora terrestre y variación orbital de Apophis', fontsize=11)

# Panel 2: componente W (la que afecta i y Ω)
axes2[1].plot(t_dias_plot, np.abs(W_comp), 'purple', linewidth=1.2, label='|W| (⊥ al plano)')
axes2[1].plot(t_dias_plot, np.abs(R_comp), 'b--',    linewidth=0.8, label='|R| (radial)',    alpha=0.7)
axes2[1].plot(t_dias_plot, np.abs(S_comp), 'g-.',    linewidth=0.8, label='|S| (transversal)', alpha=0.7)
axes2[1].axvline(t_encuentro_dias, color='red', linestyle='--', linewidth=1.5)
axes2[1].set_ylabel('Componentes RSW (AU/UT²)', fontsize=10)
axes2[1].set_yscale('log')
axes2[1].legend(fontsize=8)
axes2[1].grid(alpha=0.3)

# Panel 3: cambio en i y Ω
delta_i = np.degrees(i_nc - i_nc[0])
delta_O = np.degrees(Omega_nc - Omega_nc[0])
ax3 = axes2[2]
ax3.plot(t_dias_plot, delta_i, 'b-',  linewidth=1.5, label='$\\Delta i$ (N-cuerpos)')
ax3_twin = ax3.twinx()
ax3_twin.plot(t_dias_plot, delta_O, 'orange', linewidth=1.5, linestyle='--',
              label='$\\Delta\\Omega$ (N-cuerpos)')
ax3.axvline(t_encuentro_dias, color='red', linestyle='--', linewidth=1.5)
ax3.set_ylabel('$\\Delta i$ (°)', fontsize=10, color='b')
ax3_twin.set_ylabel('$\\Delta\\Omega$ (°)', fontsize=10, color='orange')
ax3.set_xlabel(f'Días desde {INICIO_STR}', fontsize=11)

# Leyendas combinadas
lines1, labels1 = ax3.get_legend_handles_labels()
lines2, labels2 = ax3_twin.get_legend_handles_labels()
ax3.legend(lines1 + lines2, labels1 + labels2, fontsize=8, loc='upper left')
ax3.grid(alpha=0.3)

plt.tight_layout()
plt.savefig('gauss_perturbacion_completa.png', dpi=120, bbox_inches='tight')
plt.show()
print('Figura guardada: gauss_perturbacion_completa.png')

## 9. Análisis cuantitativo y conclusiones

In [ ]:
# Cambio neto en i y Ω
delta_i_total   = np.degrees(i_nc[-1]     - i_nc[0])
delta_Omega_total = np.degrees(Omega_nc[-1] - Omega_nc[0])

# Error Gauss vs. N-cuerpos
err_i_gauss = np.degrees(np.abs(i_gauss - i_nc))
err_O_gauss = np.degrees(np.abs(Omega_gauss - Omega_nc))

print('=' * 65)
print('Análisis: Variación de elementos orbitales de Apophis')
print(f'  Ventana: {INICIO_STR} → {FIN_STR}  (±60 días del encuentro)')
print('=' * 65)

print(f'\nCambio neto de inclinación Δi   = {delta_i_total:+.6f}°')
print(f'Cambio neto de nodo Δ Ω          = {delta_Omega_total:+.6f}°')
print(f'  → El cambio es {"abrupto" if abs(delta_i_total) > 0.01 else "suave"} ')
print(f'    alrededor del día {t_encuentro_dias} (encuentro).')

print(f'\nComparación Gauss vs. N-cuerpos:')
print(f'  Error máx en i  : {err_i_gauss.max():.4e} °')
print(f'  Error máx en Ω  : {err_O_gauss.max():.4e} °')
print(f'  Error medio en i: {err_i_gauss.mean():.4e} °')

# Variables de Delaunay
a0 = a_nc[0];  e0 = e_nc[0];  i0 = i_nc[0]
L0 = np.sqrt(mu_sol_val * a0)
G0 = L0 * np.sqrt(1 - e0**2)
H0 = G0 * np.cos(i0)

a1 = a_nc[-1]; e1 = e_nc[-1]; i1 = i_nc[-1]
L1_D = np.sqrt(mu_sol_val * a1)
G1_D = L1_D * np.sqrt(1 - e1**2)
H1_D = G1_D * np.cos(i1)

print(f'\nVariables de Delaunay (formalismo canónico):')
print(f'  {"":<8} {"Inicial":>12} {"Final":>12} {"ΔPorcentaje":>14}')
print('-' * 50)
for nombre, val0, val1 in [('L', L0, L1_D), ('G', G0, G1_D), ('H', H0, H1_D)]:
    dpct = 100 * (val1 - val0) / abs(val0) if val0 != 0 else float('nan')
    print(f'  {nombre:<8} {val0:>12.6f} {val1:>12.6f} {dpct:>13.4f}%')

print('\nConclusión:')
print('  • La componente W de la perturbación es responsable de di/dt y dΩ/dt.')
print('  • Durante el encuentro, hay un salto rápido en i y Ω (no gradual).')
print('  • Las ecuaciones de Gauss reproducen bien el resultado N-cuerpos')
print('    cuando la perturbación es pequeña (lejos del encuentro).')
print('  • Cerca del encuentro, la perturbación no es pequeña y las ecuaciones')
print('    de Gauss son aproximadas (VOP de primer orden).')
print('  • En variables de Delaunay, L (~ a) cambia; H (~ cos i) también,')
print('    evidenciando el acoplamiento entre plano orbital y energía.')